In [59]:
import os
import ROOT
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.gridspec as gridspec
import multiprocessing as mp

try:
    import mplhep
    hep_style = True
    mplhep.style.use("CMS")
except ImportError:
    hep_style = False

# Define consistent process colors
colors = {
    "VZbb": "#6A0DAD",
    "VZcc": "#9B30FF",
    "VZlx": "#D8BFD8",
    "QCD": "#D3D3D3",
    "TT": "#00BFFF", 
    "ST": "#2B193D",
    "VJetcx": "#e3e146", 
    "VJetbx": "#adab10", 
    "VJetll": "#f5f768", 
    "WW": "#91bfdb",  
    "WZ_lx": "#f595f2", 
    "ZZ_lx": "#f595f2", 
    "WZ_cc": "#c734c2", 
    "ZZ_cc": "#c734c2", 
    "WZ_bb": "#73006f", 
    "ZZ_bb": "#73006f", 
    "ggZH_hbb": "#ff0000", 
    "ZH_hbb_inclusive": "#ff0000", 
    "ggZH_hcc": "#b700ff",
    "ZH_hcc": "#b700ff",
    "WH_hbb": "#ff0000", 
    "WH_hcc": "#b700ff",
}

In [60]:
def hist_to_numpy(hist):
    """Convert ROOT TH1 to numpy arrays."""
    n_bins = hist.GetNbinsX()
    bin_edges = np.array([hist.GetBinLowEdge(i+1) for i in range(n_bins)] + [hist.GetBinLowEdge(n_bins+1)])
    contents = np.array([hist.GetBinContent(i+1) for i in range(n_bins)])
    errors = np.array([hist.GetBinError(i+1) for i in range(n_bins)])
    return bin_edges, contents, errors

In [61]:
# --- ROOT file path ---
root_path = "/eos/user/h/haozhong/Combine/CMSSW_14_1_0_pre4/src/CombineHarvester/VHccCoHa_inclusive_20260506_for_preApproval/diffxsec_outputs_15_01_Gnn_SM_FR20_29jan/Run3_full/scans/breakdown/fitDiagnostics.ZHbb.root"

# --- Plot targets ---
fit_configs = [
    ("shapes_prefit", "prefit_plots/", "prefit"),
    ("shapes_fit_b", "postfit_bkg_plots/", "postfit_bkg"),
    ("shapes_fit_s", "postfit_sig_plots/", "postfit_sig"),
]

# --- Collect all tasks ---
all_tasks = []
for fit_dir, out_dir, label_suffix in fit_configs:
    file = ROOT.TFile.Open(root_path)
    shape_dir = file.Get(fit_dir)
    if not shape_dir:
        print(f"[WARN] Directory {fit_dir} not found in file. Skipping...")
        continue
    channels = [key.GetName() for key in shape_dir.GetListOfKeys()]
    os.makedirs(out_dir, exist_ok=True)
    for channel in channels:
        all_tasks.append((channel, fit_dir, out_dir, label_suffix))
    file.Close()

In [66]:

def save_stack_plot(output_dir, channel, bin_edges, stack_contents, stack_labels, stack_colors, obs_name, yerr=None, data=None, ylog=False):
    fig = plt.figure(figsize=(6, 6.5))
    gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)
    ax = fig.add_subplot(gs[0])
    ax_ratio = fig.add_subplot(gs[1], sharex=ax)

    # ------------------ MAIN STACK PLOT ------------------
    ax.hist(
        [bin_edges[:-1]] * len(stack_contents),
        bins=bin_edges,
        weights=stack_contents,
        stacked=True,
        color=stack_colors,
        edgecolor="black",
        linewidth=0.5
    )

    if yerr is not None:
        total = np.sum(stack_contents, axis=0)
        x_edges = bin_edges
        y_upper = np.append(total + yerr, total[-1] + yerr[-1])
        y_lower = np.append(total - yerr, total[-1] - yerr[-1])
        ax.fill_between(
            x_edges,
            y_lower,
            y_upper,
            step="post",
            color="gray",
            alpha=0.5,
            label="Stat. unc."
        )

    if data is not None:
        x_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
        ax.errorbar(
            x_centers,
            data,
            yerr=np.sqrt(data),
            fmt='o',
            color='black',
            markersize=4,
            label="Asimov data"
        )

    # Legend
    handles = []
    labels = []
    for label, color in zip(stack_labels, stack_colors):
        patch = plt.Rectangle((0, 0), 1, 1, facecolor=color, edgecolor='black', linewidth=0.5)
        handles.append(patch)
        labels.append(label)
    handles = handles[::-1]
    labels = labels[::-1]
    if yerr is not None:
        handles.append(plt.Rectangle((0, 0), 1, 1, facecolor="gray", alpha=0.5))
        labels.append("Stat. unc.")
    if data is not None:
        handles.append(plt.Line2D([0], [0], marker='o', color='black', linestyle='None'))
        labels.append("Asimov data")
    ax.legend(handles, labels, fontsize=9, loc="best", frameon=False)

    ax.set_ylabel("Yields", fontsize=12)
    ax.tick_params(axis='both', which='major', labelsize=12)
    if ylog:
        ax.set_yscale("log")
        ax.set_ylim(0.1, None)
    ax.set_title(f"{channel}", fontsize=10)
    
    # ax.text(0.00, 1.01, "CMS Preliminary", transform=ax.transAxes,
    #     fontsize=13, fontweight='bold', ha='left', va='top')
    # 左上角添加 "CMS Preliminary"（Preliminary 斜体小一号）
    cms_text = ax.text(0.00, 1.05, "CMS ", transform=ax.transAxes,
                       fontsize=13, fontweight='bold', ha='left', va='top')
    pre_text = ax.text(0.09, 1.045, "Preliminary", transform=ax.transAxes,
                       fontsize=11, fontstyle='italic', ha='left', va='top')
# (0.02, 0.98) 表示 axes 坐标系中的位置（x=0.02 为左侧留白，y=0.98 为
    
    ax.tick_params(labelbottom=False)

    # ------------------ RATIO PANEL ------------------
    mc_total = np.sum(stack_contents, axis=0)
    x_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
    epsilon = 1e-10
    mc_safe = np.where(mc_total == 0, epsilon, mc_total)

    # Ratio and errors
    ratio = (data - mc_total) / mc_safe
    
    # Explicitly set ratio = 0 when both data and MC are zero
    mask_zero_both = (data == 0) & (mc_total == 0)
    ratio[mask_zero_both] = np.nan

    sigma_data = np.sqrt(data)
    sigma_mc = yerr
    ratio_err = np.sqrt(
        (sigma_data / mc_safe)**2 +
        ((data - mc_total) * sigma_mc / mc_safe**2)**2
    )

    ratio_err[mask_zero_both] = 0.0  # Optional: clean up error bar

    # MC-only relative uncertainty band
    rel_mc_unc = sigma_mc / mc_safe
    # Extend band edges for step="post"
    x_edges = bin_edges
    upper_band = np.append(rel_mc_unc, rel_mc_unc[-1])
    lower_band = np.append(-rel_mc_unc, -rel_mc_unc[-1])
    ax_ratio.fill_between(
        x_edges,
        lower_band,
        upper_band,
        step="post",
        color="gray",
        alpha=0.5
    )

    # Data points with full propagated error
    ax_ratio.axhline(0, color="black", linestyle="--", linewidth=1)
    ax_ratio.errorbar(
        x_centers,
        ratio,
        yerr=ratio_err,
        fmt='o',
        color='black',
        markersize=4
    )

    # Ratio panel labels and layout
    ax_ratio.set_ylabel(r"$\frac{\mathrm{Data} - \mathrm{MC}}{\mathrm{MC}}$", fontsize=12)
    ax_ratio.yaxis.set_label_coords(-0.07, 0.5)
    ax_ratio.set_xlabel(obs_name, fontsize=12)
    ax_ratio.tick_params(axis='both', which='major', labelsize=11)
    ax_ratio.set_ylim(-2.1, 2.1)
    ax_ratio.grid(True, axis='y', linestyle='--', alpha=0.5)

    # Layout
    fig.subplots_adjust(hspace=0.05, top=0.95, bottom=0.12, left=0.15, right=0.95)

    suffix = "_log" if ylog else "_lin"
    for ext in ["pdf"]: #, "png"
        save_path = os.path.join(output_dir, f"{channel}_{suffix}.{ext}")
        plt.savefig(save_path, dpi=300, transparent=True)

    plt.close()


In [67]:
def process_channel(channel, shape_dir_name, output_dir, label_suffix):
    file = ROOT.TFile.Open(root_path)
    shape_dir = file.Get(shape_dir_name)
    channel_dir = shape_dir.Get(channel)

    stack_contents = []
    stack_labels = []
    stack_colors = []
    bin_edges = None

    for process, color in colors.items():
        hist = channel_dir.Get(process)
        if not hist:
            continue
        be, contents, errors = hist_to_numpy(hist)
        if bin_edges is None:
            bin_edges = be
        stack_contents.append(contents)
        stack_labels.append(process)
        stack_colors.append(color)

    stack_contents = stack_contents[::-1]
    stack_labels = stack_labels[::-1]
    stack_colors = stack_colors[::-1]

    if shape_dir_name == "shapes_fit_b":
        # Get background-only total
        shape_dir = file.Get("shapes_fit_s")
        _channel_dir = shape_dir.Get(channel)
        total_hist = _channel_dir.Get("total")
    else:
        total_hist = channel_dir.Get("total")

    yerr = None
    data = None
    if total_hist:
        _, total_content, total_errors = hist_to_numpy(total_hist)
        data = total_content
        if np.any(total_content):
            last_nonzero_bin = np.max(np.nonzero(total_content)[0])
            trim_index = last_nonzero_bin + 1
            bin_edges = bin_edges[:trim_index + 1]
            stack_contents = [content[:trim_index] for content in stack_contents]
            yerr = total_errors[:trim_index]
            data = data[:trim_index]
        else:
            print(f"[WARN] All bins in 'total' are zero for channel: {channel}")
            yerr = None
            data = None

    obs_name = "bins number"
    if '1_13p6TeV' in channel or '2_13p6TeV' in channel:
        obs_name = "ML score [score bins]"

    save_stack_plot(
        output_dir=output_dir,
        channel=f"{channel}_{label_suffix}",
        bin_edges=bin_edges,
        stack_contents=stack_contents,
        stack_labels=stack_labels,
        stack_colors=stack_colors,
        obs_name=obs_name,
        yerr=yerr,
        data=data,
        ylog=False
    )
    save_stack_plot(
        output_dir=output_dir,
        channel=f"{channel}_{label_suffix}",
        bin_edges=bin_edges,
        stack_contents=stack_contents,
        stack_labels=stack_labels,
        stack_colors=stack_colors,
        obs_name=obs_name,
        yerr=yerr,
        data=data,
        ylog=True
    )

    file.Close()


In [68]:
from tqdm import tqdm
from functools import partial
def process_channel_wrapper(args):
    return process_channel(*args)


In [69]:
if __name__ == "__main__":
    with mp.Pool(processes=10) as pool:
        for _ in tqdm(pool.imap_unordered(process_channel_wrapper, all_tasks), total=len(all_tasks)):
            pass

100%|██████████| 30/30 [00:36<00:00,  1.21s/it]
